In [ ]:
import pandas as pd
import numpy as np
import textstat
import string
import spacy
import re

import seaborn as sns
import matplotlib.pyplot as plt

import pyphen
from wordfreq import zipf_frequency

import math
from collections import Counter
import networkx as nx

import nltk
nltk.download('cmudict')
textstat.set_lang("pt")
from nltk.corpus import wordnet as wn

[nltk_data] Downloading package cmudict to /home/luan/nltk_data...
[nltk_data]   Package cmudict is already up-to-date!


## (1) Functions and Processing

In [ ]:
# Surface Features

nlp = spacy.load("pt_core_news_lg")

def n_words(text):
    doc = nlp(text)
    words = [token for token in doc if not token.is_space]
    return max(len(words), 1)

def n_sentences(text):
    doc = nlp(text)
    sentences = list(doc.sents)
    return max(len(sentences), 1)

def words_per_sentence(text):
    return n_words(text) / n_sentences(text)

def n_characters(text):
    return len(text.replace(" ", ""))

def reading_time(text, wpm=260):
    words = n_words(text)
    return words / wpm

def punctuation_features(text):
    punct_set = set(string.punctuation)

    total_punct = sum(1 for ch in text if ch in punct_set)
    n_w = n_words(text)
    n_s = n_sentences(text)

    exclam = text.count("!")
    quest = text.count("?")

    return {
        "total_punct": total_punct,
        "punct_per_word": total_punct / n_w,
        "punct_per_sentence": total_punct / n_s,
        "exclam": exclam,
        "quest": quest
    }

def uppercase_ratio(text):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters:
        return 0.0
    
    upper = sum(1 for ch in letters if ch.isupper())
    return upper / len(letters)

def char_repetition_ratio(text):
    pattern = re.compile(r"(.)\1{2,}") # 3+

    words = text.split()
    if not words:
        return 0.0
    
    repeated = sum(1 for w in words if pattern.search(w.lower()))
    return repeated / len(words)

def avg_letters_per_word(text):
    words = re.findall(r'\b\w+\b', text, flags=re.UNICODE)
    if not words:
        return 0.0

    total_letters = sum(len(w) for w in words)
    return total_letters / len(words)

dic = pyphen.Pyphen(lang='pt_BR')
def avg_syllables_per_word(text):
    words = re.findall(r'\b\w+\b', text, flags=re.UNICODE)
    if not words:
        return 0.0

    syllables = sum(len(dic.inserted(w).split('-')) for w in words)
    return syllables / len(words)

In [ ]:
# Lexical Features

def lexical_frequency(text):
    doc = nlp(text)
    freqs = []

    for token in doc:
        if token.is_alpha:
            freq = zipf_frequency(token.text.lower(), 'pt')
            freqs.append(freq)

    if not freqs:
        return 0.0
    
    return {
        "rarest_word": min(freqs),
        "most_common_word": max(freqs),
        "avg_freq": np.mean(freqs),
        "freq_disp": np.std(freqs)
    }

def ttr(text):
    doc = nlp(text)
    tokens = [t.text.lower() for t in doc if t.is_alpha]

    if not tokens:
        return 0.0
    
    unique = set(tokens)
    return len(unique) / len(tokens)

def stopword_ratio(text):
    doc = nlp(text)

    words = [t for t in doc if t.is_alpha]
    if not words:
        return 0.0

    stopwords = [t for t in words if t.is_stop]
    
    return len(stopwords) / len(words)

def n_unique_words(text):
    doc = nlp(text)
    tokens = [t.text.lower() for t in doc if t.is_alpha]

    return len(set(tokens))

def mean_unique_word_length(text):
    doc = nlp(text)
    tokens = [t.text.lower() for t in doc if t.is_alpha]

    unique = set(tokens)
    if not unique:
        return 0.0

    return np.mean([len(w) for w in unique])

def count_syllables(word):
    return len(dic.inserted(word).split('-'))

def flesch_portuguese(text):
    doc = nlp(text)

    sentences = list(doc.sents)
    words = [t.text for t in doc if t.is_alpha]

    if not sentences or not words:
        return 0.0

    n_sent = len(sentences)
    n_words = len(words)
    n_syll = sum(count_syllables(w) for w in words)

    flesch = 226 - 1.04 * (n_words / n_sent) - 72 * (n_syll / n_words)
    return flesch

In [ ]:
# Syntactic Features

POS_TAGS = [
    "NOUN", "VERB", "ADJ", "ADV", "PRON", "DET",
    "ADP", "AUX", "CCONJ", "SCONJ", "NUM", "PROPN", "INTJ"
]

def pos_tag_counts(text):
    doc = nlp(text)

    counts = {pos: 0 for pos in POS_TAGS}

    for token in doc:
        if token.pos_ in counts:
            counts[token.pos_] += 1
    
    return counts

def mean_verbs_per_sentence(text):
    doc = nlp(text)

    sentences = list(doc.sents)
    if len(sentences) == 0:
        return 0.0

    verb_counts = []

    for sent in sentences:
        verbs = sum(1 for tok in sent if tok.pos_ == "VERB")
        verb_counts.append(verbs)

    return np.mean(verb_counts)

def mean_aux_per_sentence(text):
    doc = nlp(text)

    sentences = list(doc.sents)
    if len(sentences) == 0:
        return 0.0

    aux_counts = []

    for sent in sentences:
        aux = sum(1 for tok in sent if tok.pos_ == "AUX")
        aux_counts.append(aux)

    return np.mean(aux_counts)

def total_auxiliaries(text, nlp):
    doc = nlp(text)
    return sum(1 for tok in doc if tok.pos_ == "AUX")

def tree_depth(token):
    if not list(token.children):
        return 1
    return 1 + max(tree_depth(child) for child in token.children)

def mean_dependency_tree_depth(text):
    doc = nlp(text)

    depths = []

    for sent in doc.sents:
        root = sent.root
        depths.append(tree_depth(root))

    if len(depths) == 0:
        return 0.0

    return np.mean(depths)

def dependency_relation_counts(text):
    doc = nlp(text)

    dep_counts = {}

    for token in doc:
        dep = token.dep_
        dep_counts[dep] = dep_counts.get(dep, 0) + 1

    return dep_counts

def mean_dependency_length(text):
    doc = nlp(text)

    distances = []

    for token in doc:
        if token.head != token:
            distance = abs(token.i - token.head.i)
            distances.append(distance)

    if len(distances) == 0:
        return 0.0

    return np.mean(distances)

SUBORDINATE_DEPS = {"advcl", "ccomp", "xcomp", "acl", "relcl"}

def mean_subordinate_clauses(text):
    doc = nlp(text)

    sentences = list(doc.sents)
    if len(sentences) == 0:
        return 0.0

    counts = []

    for sent in sentences:
        sub_count = sum(
            1 for tok in sent
            if tok.dep_ in SUBORDINATE_DEPS
        )
        counts.append(sub_count)

    return np.mean(counts)

In [ ]:
# Other Features (Discourse and Semantics)

def lexical_ambiguity(text):
    doc = nlp(text)

    ambiguities = []

    for token in doc:
        if token.is_alpha:
            synsets = wn.synsets(token.text, lang='por')
            ambiguities.append(len(synsets))

    if (len(ambiguities) == 0):
        return 0.0

    return sum(ambiguities) / len(ambiguities)

def lexical_entropy(text):
    doc = nlp(text)

    tokens = [t.lemma_.lower() for t in doc if t.is_alpha]

    if len(tokens) == 0:
        return 0.0
    
    counts = Counter(tokens)
    total = len(tokens)

    entropy = 0.0
    for count in counts.values():
        p = count / total
        entropy -= p * math.log(p)
    
    return entropy

CONNECTIVES_PT = {
    "porque", "pois", "mas", "porém", "entretanto",
    "logo", "portanto", "assim", "além disso",
    "embora", "se", "quando", "enquanto"
}

def cohesion_ratio(text):
    doc = nlp(text)

    tokens = [t for t in doc if t.is_alpha]
    if len(tokens) == 0:
        return 0.0

    count = sum(
        1 for t in tokens
        if t.text.lower() in CONNECTIVES_PT
    )

    return count / len(tokens)

DISCOURSE_MARKERS = {
    "então", "assim", "portanto", "porém",
    "logo", "ou seja", "bem", "agora"
}

def discourse_markers_ratio(text):
    doc = nlp(text)

    tokens = [t for t in doc if t.is_alpha]
    if len(tokens) == 0:
        return 0.0
    
    count = sum(
        1 for token in doc
        if token.text.lower() in DISCOURSE_MARKERS
    )

    return count / len(tokens)

def children_dependency_tree(text):
    doc = nlp(text)

    children_counts = [
        len(list(token.children))
        for token in doc
    ]

    if len(children_counts) == 0:
        return 0.0

    return {
        "mean_children_per_token": sum(children_counts) / len(children_counts),
        "max_children": max((len(list(token.children)) for token in doc), default=0),
        "min_children": min((len(list(token.children)) for token in doc), default=0),
    }
    
def root_verb_position(text):
    doc = nlp(text)

    positions = []

    for sent in doc.sents:
        tokens = list(sent)
        if len(tokens) == 0:
            continue

        root = sent.root

        pos = (root.i - sent.start) / len(tokens)
        positions.append(pos)

    if len(positions) == 0:
        return 0.0
    
    return np.mean(positions)

def dependency_pattern_diversity(text):
    doc = nlp(text)

    patterns = set(
        (t.head.pos_, t.dep_, t.pos_)
        for t in doc
    )

    return len(patterns)

def subtree_size(text, nlp):
    doc = nlp(text)

    sizes = [
        len(list(token.subtree))
        for token in doc
    ]

    if len(sizes) == 0:
        return 0.0

    return {
        "mean_subtree_size": np.mean(sizes),
        "max_subtree_size": max((len(list(token.subtree)) for token in doc), default=0),
        "min_subtree_size": min((len(list(token.subtree)) for token in doc), default=0),
    }

def named_entity_ratio(text):
    doc = nlp(text)

    tokens = [t for t in doc if t.is_alpha]
    if len(tokens) == 0:
        return 0.0

    return len(doc.ents) / len(tokens)

def entity_type_diversity(text):
    doc = nlp(text)
    return len(set(ent.label_ for ent in doc.ents))

def dependency_graph_metrics(text):
    doc = nlp(text)

    G = nx.Graph()

    for token in doc:
        G.add_node(token.i)
        if token.head != token:
            G.add_edge(token.i, token.head.i)

    if len(G.nodes) == 0:
        return {
            "avg_degree": 0.0,
            "density": 0.0,
            "clustering": 0.0
        }

    avg_degree = np.mean([d for _, d in G.degree()])
    density = nx.density(G)
    clustering = nx.average_clustering(G)

    return {
        "avg_degree": avg_degree,
        "density": density,
        "clustering": clustering
    }

def passive_voice_ratio(text):
    doc = nlp(text)

    sentences = list(doc.sents)

    if len(sentences) == 0:
        return 0.0

    passive_count = 0

    for sent in sentences:
        if any("pass" in token.dep_ for token in sent):
            passive_count += 1

    return passive_count / len(sentences)

## (2) Features for GoEmotions

In [12]:
df = pd.read_csv("../data/treated/go_emotions_treated.csv")

df['UNCLEAN_TEXT_PT'] = (
    df['UNCLEAN_TEXT_PT']
    .fillna("")        # remove NaN
    .astype(str)       # garante string
)

print(df.shape)
df.head(3)

(54234, 41)


,id,admiration,amusement,anger,annoyance,approval,caring,confusion,curiosity,desire,...,UNCLEAN_TEXT_PT,BASE_TEXT_PT,TEXT_NO_STOP_PT,TEXT_LEMMA_PT,CLEAN_TEXT_PT,UNCLEAN_TEXT_EN,BASE_TEXT_EN,TEXT_NO_STOP_EN,TEXT_LEMMA_EN,CLEAN_TEXT_EN
0,eczazk6,0,0,0,0,1,0,0,0,0,...,Tão rápido quanto [NOME] me carregará. Seriame...,tao rapido quanto nome carregara seriamente up...,tao rapido nome carregara seriamente uptown ce...,tao rapido quanto nome carregar seriamente upt...,tao rapido nome carregar seriamente uptown cen...,Fast as [NAME] will carry me. Seriously uptown...,fast name will carry seriously uptown downtown...,fast name will carry seriously uptown downtown...,fast name will carry seriously uptown downtown...,fast name will carry seriously uptown downtown...
1,eczb07q,0,0,0,0,0,0,0,0,0,...,Você estragou isso. Eles tocaram você como um ...,voce estragou isso eles tocaram voce como violino,voce estragou tocaram voce violino,voce estragar isso eles tocar voce como violino,voce estragar tocar voce violino,You blew it. They played you like a fiddle.,you blew they played you like fiddle,you blew they played you like fiddle,you blew they played you like fiddle,you blew they played you like fiddle
2,eczb4bm,0,0,0,0,0,0,0,0,0,...,TL;DR Não há mais Super Bowls para [NAME]. Pre...,nao mais super bowls para name prepare-se para...,nao super bowls name prepare-se temporada vito...,nao mais super bowl para name preparar se para...,nao super bowl name preparar se temporada vito...,TL;DR No more Superbowls for [NAME]. Get ready...,more superbowls for name get ready for another...,more superbowls name get ready another winning...,more superbowls ser name get ready ser another...,more superbowls name get ready another winning...


In [ ]:
# unclean_text_col = 'UNCLEAN_TEXT_PT'

# textstat_feats = get_textstat_features(df[unclean_text_col].tolist()) 

# textstat_cols = [
#     "letter_count",
#     "sentence_count",
#     "avg_letter_per_word",
#     "words_per_sentence",
#     "avg_syllables_per_word",
#     "reading_time",
# ]

# df_textstat = pd.DataFrame(textstat_feats, columns=textstat_cols) 
# df = pd.concat([df.reset_index(drop=True), df_textstat.reset_index(drop=True)], axis=1)

# punct_feats = mean_punctuation_per_word(df[unclean_text_col].tolist())

# punct_cols = [
#     "punctuation_per_word",
#     "exclamation_per_word",
#     "question_per_word"
# ]

# df_punct = pd.DataFrame(punct_feats, columns=punct_cols)

# df = pd.concat([df.reset_index(drop=True), df_punct], axis=1)

# df["mean_verbs_per_word"] = mean_verbs_per_word(df[unclean_text_col])

# df["mean_auxiliaries_per_word"] = mean_auxiliaries_per_word(df[unclean_text_col].tolist())
# df["mean_auxiliaries_per_sentence"] = mean_auxiliaries_per_sentence(df[unclean_text_col].tolist())

# df["flesch_portuguese"] = flesch_portuguese(df[unclean_text_col].tolist())

# df["uppercase_percentage"] = uppercase_percentage(df[unclean_text_col].tolist())
# df["mean_unique_word_length"] = mean_unique_word_length(df[unclean_text_col].tolist())
# df["mean_char_repetition_per_word"] = mean_char_repetition_per_word(df[unclean_text_col].tolist())

In [ ]:
emotion_cols = [
    'admiration', 'amusement', 'anger', 'annoyance',
    'approval', 'caring', 'confusion', 'curiosity', 'desire',
    'disappointment', 'disapproval', 'disgust', 'embarrassment',
    'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love',
    'nervousness', 'optimism', 'pride', 'realization', 'relief',
    'remorse', 'sadness', 'surprise', 'neutral'
]

num_cols = df.select_dtypes(include="number")
num_cols = [col for col in num_cols if col not in emotion_cols]
df_num = df[num_cols]
corr = df_num.corr(method="pearson")

mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(16, 12))
sns.heatmap(
    corr,
    mask=mask,
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)

plt.title("Correlation Matrix (Pearson)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
df.columns

In [ ]:
emotion_cols = [
    'admiration', 'amusement', 'anger', 'annoyance',
    'approval', 'caring', 'confusion', 'curiosity', 'desire',
    'disappointment', 'disapproval', 'disgust', 'embarrassment',
    'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love',
    'nervousness', 'optimism', 'pride', 'realization', 'relief',
    'remorse', 'sadness', 'surprise', 'neutral'
]

base_cols = ["UNCLEAN_TEXT_PT", "BASE_TEXT_PT"]
remove_cols = ['TEXT_NO_STOP', 'CLEAN_TEXT', 'TEXT_LEMMA', 'texto']

feature_cols = [c for c in df.columns if c not in base_cols and c not in remove_cols and c not in emotion_cols]

df_final = df[base_cols + emotion_cols + feature_cols]

df_final.columns

In [20]:
df_final.to_csv(
    "../data/feat/go_emotions_with_features.csv",
    index=False
)